# Exploring change_ds — Why Are Some Std Devs So Large?

Replicates the `change_ds` construction from `analysis_app.py` and converts to a flat DataFrame for inspection.

In [ ]:
import os, sys

# Source paths are all relative to the repo root — set cwd there
REPO_ROOT = os.path.dirname(os.path.abspath(''))   # notebooks/ -> repo root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(REPO_ROOT)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
import xarray as xr

print("Working directory:", os.getcwd())

In [ ]:
from c_Processing.c_main_data_pipeline import create_full_underived_df, to_dataset
from d_Transformations.calc_changes import calc_period_over_period_change

# Match defaults from analysis_app.py
selected_states = ['ME']
num_years_ma = 5

underived_df = create_full_underived_df(selected_states)
underived_ds = to_dataset(underived_df)

change_ds = calc_period_over_period_change(underived_ds, 'value', num_years_ma)
print(change_ds)

## Convert to DataFrame

In [ ]:
df = change_ds.to_dataframe().reset_index()
print(df.shape)
df.head(10)

## Std dev by measure — spot large values

In [ ]:
pct_std = (
    df.dropna(subset=['pct_change'])
    .groupby('measure')['pct_change']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'mean': 'mean_pct_change', 'std': 'std_pct_change'})
    .sort_values('std_pct_change', ascending=False)
)
pct_std.head(20)

## Drill into a high-std measure — show the extreme rows

In [ ]:
top_measure = pct_std.index[0]
print(f'Inspecting: {top_measure}')

sub = df[df['measure'] == top_measure].dropna(subset=['pct_change'])
sub_sorted = sub.reindex(sub['pct_change'].abs().sort_values(ascending=False).index)
sub_sorted[['organization', 'state', 'year', 'value', 'ln_value', 'pct_change', 'ma_pct_change']].head(30)

## Check for near-zero denominators (small base values driving large % changes)

In [ ]:
# Rows where pct_change is extreme but the prior-year value was tiny
# (prior value = current value / (1 + pct_change))
df['prior_value'] = df['value'] / (1 + df['pct_change'])

extreme = df[df['pct_change'].abs() > 1].copy()  # > 100% change
extreme_sorted = extreme.sort_values('pct_change', key=abs, ascending=False)
extreme_sorted[['organization', 'state', 'measure', 'year', 'prior_value', 'value', 'pct_change']].head(40)

## Summary: how many extreme observations per measure?

In [ ]:
extreme_counts = (
    extreme.groupby('measure')
    .size()
    .rename('n_extreme_obs')
    .sort_values(ascending=False)
)
extreme_counts.head(20)